# Pipeline: damage_classification (modelo 2 - tipo de dano + ubicacion, formato YOLO)

Lee los XML de `raw/`, arma las cajas por tipo de dano, y genera `images/`, `labels/`, `classes.txt` y `data.yaml` en `processed/damage_classification/` (sin split train/val/test).

In [ ]:
import sys
sys.path.append('..')

from pathlib import PurePosixPath
from extract.extract_annotations import list_xml_keys, download_xml
from transform.parse_annotations import parse_yolo_annotation, to_yolo_line
from load.load_to_s3 import copy_image, upload_bytes

BUCKET = "humanoid-damage-cnn-834257582282-us-east-1-an"
RAW_ANNOTATIONS_PREFIX = "raw/IDEA/damages/annotations/"
RAW_IMAGES_PREFIX = "raw/IDEA/damages/images/"
PROCESSED_PREFIX = "processed/damage_classification/"
DRY_RUN = True  # cambiar a False cuando quieras subir de verdad

## Extract

In [ ]:
xml_keys = list_xml_keys(BUCKET, RAW_ANNOTATIONS_PREFIX)
print(f"Encontrados {len(xml_keys)} archivos XML.")

## Transform

In [ ]:
parsed = []
for key in xml_keys:
    try:
        xml_bytes = download_xml(BUCKET, key)
        annotation = parse_yolo_annotation(xml_bytes)
    except Exception as exc:
        print(f"[ERROR] {key}: {exc}")
        continue
    if not annotation["image_basename"]:
        print(f"[ERROR] {key}: no se pudo determinar el nombre de la imagen")
        continue
    if not annotation["objects"]:
        continue  # sin daños anotados, no aporta a este dataset
    parsed.append(annotation)

print(f"Anotaciones con objetos: {len(parsed)}")

In [ ]:
all_damage_types = sorted({obj["damage_type"] for a in parsed for obj in a["objects"]})
class_to_id = {name: i for i, name in enumerate(all_damage_types)}
print(f"Tipos de dano ({len(all_damage_types)}): {all_damage_types}")

## Load

In [ ]:
upload_bytes(BUCKET, f"{PROCESSED_PREFIX}classes.txt", ("\n".join(all_damage_types) + "\n").encode("utf-8"), dry_run=DRY_RUN)

data_yaml = (
    "train: images\n"
    f"nc: {len(all_damage_types)}\n"
    f"names: {all_damage_types}\n"
)
upload_bytes(BUCKET, f"{PROCESSED_PREFIX}data.yaml", data_yaml.encode("utf-8"), dry_run=DRY_RUN)

In [ ]:
for i, annotation in enumerate(parsed, start=1):
    basename = annotation["image_basename"]
    source_key = f"{RAW_IMAGES_PREFIX}{basename}"

    dest_image_key = f"{PROCESSED_PREFIX}images/{basename}"
    copy_image(BUCKET, source_key, dest_image_key, dry_run=DRY_RUN)

    label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
    label_key = f"{PROCESSED_PREFIX}labels/{PurePosixPath(basename).stem}.txt"
    upload_bytes(BUCKET, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=DRY_RUN)

    if i % 50 == 0 or i == len(parsed):
        print(f"Procesadas {i}/{len(parsed)} imagenes...")

print("Listo." if not DRY_RUN else "[DRY RUN] no se subio nada, poner DRY_RUN = False para ejecutar de verdad.")